In [6]:
# Клонируем репозиторий CatVTON
!git clone https://github.com/Zheng-Chong/CatVTON.git
%cd CatVTON

fatal: destination path 'CatVTON' already exists and is not an empty directory.
/content/CatVTON


In [9]:
# Устанавливаем зависимости из requirements.txt
!pip install -r requirements.txt

# Обновляем accelerate до последней версии (>=0.32.0) и ставим стабильный diffusers
!pip install -U accelerate diffusers==0.30.3 peft huggingface_hub

  Cloning https://github.com/huggingface/diffusers.git to /tmp/pip-req-build-0uv7n_8e
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers.git /tmp/pip-req-build-0uv7n_8e
  Resolved https://github.com/huggingface/diffusers.git to commit c8eba433adf1f90d7fcc70092562ea50789ee8fb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached accelerate-0.31.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-0.31.0-py3-none-any.whl (309 kB)
  Created wheel for diffusers: filename=diffusers-0.38.0.dev0-py3-none-any.whl size=5245992 sha256=19d3b31b271cab73edb468e908f111ec7eaf371f81ca98ae53258752791ff1ca
  Stored in directory: /tmp/pip-ephem-wheel-cache-kmb5or6y/wheels/23/0f/7d/f97813d265ed0e599a78d83afd4e1925740896ca79b46cccfd
Successfully built diffusers
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.30.3
    Uninstalling 

  Using cached diffusers-0.30.3-py3-none-any.whl.metadata (18 kB)
Using cached diffusers-0.30.3-py3-none-any.whl (2.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 34.8 MB/s eta 0:00:00
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.38.0.dev0
    Uninstalling diffusers-0.38.0.dev0:
      Successfully uninstalled diffusers-0.38.0.dev0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.31.0
    Uninstalling accelerate-0.31.0:
      Successfully uninstalled accelerate-0.31.0


In [5]:
import urllib.request

# Скачиваем тестовое фото человека (вид спереди)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/Zheng-Chong/CatVTON/main/resource/demo/example/person/men/model_5.png",
    "person.png"
)

# Скачиваем тестовое фото одежды
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/Zheng-Chong/CatVTON/main/resource/demo/example/condition/upper/24083449_54173465_2048.jpg",
    "garment.jpg"
)

('garment.jpg', <http.client.HTTPMessage at 0x7c06082fc110>)

In [2]:
%cd CatVTON

/content/CatVTON


In [3]:
%ls

app_flux.py  detectron2/   LICENSE                      README.md
app_p2p.py   eval.py       model/                       requirements.txt
app.py       index.html    preprocess_agnostic_mask.py  resource/
densepose/   inference.py  __pycache__/                 utils.py


In [16]:
import torch
from PIL import Image
from huggingface_hub import snapshot_download

# imports from CatVTON
from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline

In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [14]:
weight_dtype = torch.bfloat16

# Используем официальную базовую модель
base_model_path = "runwayml/stable-diffusion-inpainting"
catvton_model_path = "zhengchong/CatVTON"

# Инициализация пайплайна правильным способом
pipeline = CatVTONPipeline(
    base_ckpt=base_model_path,
    attn_ckpt=catvton_model_path,
    attn_ckpt_version="mix",
    weight_dtype=weight_dtype,
    device=device,
    skip_safety_check=True
)

try:
    pipeline.enable_xformers_memory_efficient_attention()
except Exception:
    print("xformers не установлен, пропускаем оптимизацию памяти.")

scheduler_config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/748 [00:00<?, ?B/s]

An error occurred while trying to fetch runwayml/stable-diffusion-inpainting: runwayml/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


unet/diffusion_pytorch_model.bin:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Downloaded zhengchong/CatVTON to /root/.cache/huggingface/hub/models--zhengchong--CatVTON/snapshots/2969fcf85fe62f2036605716f0b56f0b81d01d79
xformers не установлен, пропускаем оптимизацию памяти.


AssertionError: Config file 'zhengchong/DensePose/densepose_rcnn_R_50_FPN_s1x.yaml' does not exist!

In [18]:
import os
from huggingface_hub import snapshot_download

# Скачиваем основной репозиторий с весами (или берем из кэша)
repo_path = snapshot_download(repo_id="zhengchong/CatVTON")

# Инициализация модуля для автоматического создания масок с использованием путей внутри основного репозитория
automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, "DensePose"),
    schp_ckpt=os.path.join(repo_path, "SCHP"),
    device=device
)

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/content/CatVTON/model/SCHP/__init__.py:93: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(ckpt_path, map_location='cpu')['state_dict']


AutoMasker успешно инициализирован!
